# Capítulo 5 · Optimización de Hiperparámetros con GridSearchCV

## Diplomado en Data Engineering

### Laboratorio explicativo

En los cuadernos anteriores estudiamos:

- **Overfitting y generalización**.
- **Hold-Out y K-Fold Cross Validation**.
- La importancia de evaluar un modelo sobre datos no utilizados durante su entrenamiento.

Ahora utilizaremos esos conceptos para responder una nueva pregunta:

> **¿Cómo encontramos una configuración adecuada para un modelo sin probar manualmente cada alternativa?**

Para ello utilizaremos `GridSearchCV`, una herramienta de Scikit-learn que combina:

1. Una lista de hiperparámetros posibles.
2. Validación cruzada K-Fold.
3. Una métrica de evaluación.
4. La selección automática de la mejor configuración.

## Objetivos del laboratorio

Al finalizar el cuaderno, el estudiante será capaz de:

- Diferenciar parámetros e hiperparámetros.
- Comprender qué controla un hiperparámetro.
- Analizar el efecto de `max_depth` en un árbol de decisión.
- Explicar el funcionamiento interno de Grid Search.
- Utilizar `GridSearchCV`.
- Interpretar `best_params_`, `best_score_` y `cv_results_`.
- Evaluar el mejor modelo sobre un conjunto de prueba.
- Entrenar el modelo final con todos los datos disponibles.

# 1. Parámetros e hiperparámetros

## Parámetros

Los **parámetros** son valores que el modelo aprende automáticamente durante el entrenamiento.

Ejemplos:

- Coeficientes de una regresión lineal.
- Puntos de corte de un árbol.
- Pesos de una red neuronal.

El usuario no asigna directamente estos valores. El algoritmo los calcula a partir de los datos.

## Hiperparámetros

Los **hiperparámetros** son configuraciones definidas antes del entrenamiento. Controlan cómo aprende el modelo, su complejidad y su comportamiento.

| Modelo | Ejemplo de hiperparámetro | ¿Qué controla? |
|---|---|---|
| Árbol de decisión | `max_depth` | Profundidad máxima |
| Random Forest | `n_estimators` | Cantidad de árboles |
| KNN | `n_neighbors` | Cantidad de vecinos |
| SVM | `C` | Intensidad de la regularización |

En este laboratorio utilizaremos un **Árbol de Decisión para regresión**, porque sus hiperparámetros son fáciles de interpretar.

## 2. Importación de bibliotecas

In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error

RANDOM_STATE = 42

## 3. Carga del dataset

In [6]:
df = pd.read_csv("salarios.csv")
df.head()

,AñosExperiencia,NivelEducacion,Puntaje,Salario
0,6.24,3,73.08,58436.10
1,14.31,4,78.02,100542.58
2,11.25,3,74.65,76953.98
3,9.38,1,63.31,62497.39
4,3.18,4,86.43,54024.34


## 4. Preparación de las variables

Utilizaremos como variables predictoras:

- Años de experiencia.
- Nivel de educación.
- Puntaje.

La variable objetivo será el salario.

In [7]:
X = df[
    [
        "AñosExperiencia",
        "NivelEducacion",
        "Puntaje"
    ]
]

y = df["Salario"]

print(f"Observaciones: {len(df)}")
print(f"Variables predictoras: {X.shape[1]}")

display(X.head())

Observaciones: 118
Variables predictoras: 3


,AñosExperiencia,NivelEducacion,Puntaje
0,6.24,3,73.08
1,14.31,4,78.02
2,11.25,3,74.65
3,9.38,1,63.31
4,3.18,4,86.43


# 5. Separación entre entrenamiento y prueba

Antes de buscar hiperparámetros reservaremos un **20% de los datos como conjunto de prueba**.

Este conjunto no participará en:

- La búsqueda de hiperparámetros.
- La validación cruzada.
- La elección de la mejor configuración.

Su función será realizar una evaluación final e independiente.

> **Regla importante:** el conjunto de prueba no debe utilizarse para tomar decisiones durante el desarrollo del modelo.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f"Entrenamiento: {len(X_train)} observaciones")
print(f"Prueba: {len(X_test)} observaciones")

Entrenamiento: 94 observaciones
Prueba: 24 observaciones


# 6. Modelo base

Primero entrenaremos un árbol sin ajustar manualmente sus hiperparámetros.

Este modelo servirá como referencia inicial.

Un árbol sin restricciones puede crecer hasta memorizar gran parte del conjunto de entrenamiento, por lo que puede presentar overfitting.

In [9]:
modelo_base = DecisionTreeRegressor(
    random_state=RANDOM_STATE
)

modelo_base.fit(X_train, y_train)

pred_train_base = modelo_base.predict(X_train)
pred_test_base = modelo_base.predict(X_test)

r2_train_base = r2_score(y_train, pred_train_base)
r2_test_base = r2_score(y_test, pred_test_base)
mae_test_base = mean_absolute_error(y_test, pred_test_base)

print("MODELO BASE")
print(f"R² entrenamiento: {r2_train_base:.3f}")
print(f"R² prueba: {r2_test_base:.3f}")
print(f"MAE prueba: {mae_test_base:,.0f}")
print(f"Profundidad alcanzada: {modelo_base.get_depth()}")

MODELO BASE
R² entrenamiento: 1.000
R² prueba: 0.603
MAE prueba: 3,698
Profundidad alcanzada: 12


## Interpretación del modelo base

Un resultado muy alto en entrenamiento no garantiza que el modelo generalice.

Debemos comparar:

- **R² de entrenamiento:** qué tan bien reproduce los datos utilizados para aprender.
- **R² de prueba:** qué tan bien funciona con observaciones nuevas.

Cuando el R² de entrenamiento es mucho mayor que el R² de prueba, existe una señal de sobreajuste.

Una forma de controlar la complejidad del árbol es limitar su profundidad mediante `max_depth`.

# 7. ¿Qué controla `max_depth`?

`max_depth` indica la cantidad máxima de niveles que puede tener el árbol.

- Profundidad muy baja: modelo demasiado simple, posible **underfitting**.
- Profundidad intermedia: equilibrio entre aprendizaje y generalización.
- Profundidad muy alta: modelo complejo, posible **overfitting**.

Antes de utilizar Grid Search, evaluaremos manualmente algunas profundidades para comprender qué se está optimizando.

In [10]:
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

resultados_profundidad = []

for profundidad in range(1, 16):

    modelo = DecisionTreeRegressor(
        max_depth=profundidad,
        random_state=RANDOM_STATE
    )

    scores = cross_val_score(
        modelo,
        X_train,
        y_train,
        cv=kfold,
        scoring="r2"
    )

    resultados_profundidad.append({
        "max_depth": profundidad,
        "R² promedio CV": scores.mean(),
        "Desviación estándar": scores.std()
    })

df_profundidades = pd.DataFrame(resultados_profundidad)

display(
    df_profundidades.style.format({
        "R² promedio CV": "{:.3f}",
        "Desviación estándar": "{:.3f}"
    })
)

,max_depth,R² promedio CV,Desviación estándar
0,1,0.655,0.003
1,2,0.864,0.085
2,3,0.917,0.078
3,4,0.940,0.074
4,5,0.943,0.079
5,6,0.944,0.076
6,7,0.944,0.072
7,8,0.942,0.075
8,9,0.945,0.074
9,10,0.943,0.073


## ¿Cómo interpretar esta tabla?

Cada fila representa un valor diferente de `max_depth`.

Para cada profundidad:

1. Se construye un árbol.
2. Se evalúa mediante K-Fold con K = 5.
3. Se obtienen cinco valores de R².
4. Se calcula su promedio.
5. Se calcula su desviación estándar.

El mejor valor no se selecciona por el resultado de un único Fold, sino por el **R² promedio de validación cruzada**.

In [11]:
fig = px.line(
    df_profundidades,
    x="max_depth",
    y="R² promedio CV",
    markers=True,
    title="R² promedio según la profundidad del árbol"
)

fig.update_layout(
    height=500,
    xaxis_title="Profundidad máxima",
    yaxis_title="R² promedio de validación cruzada"
)

fig.show()

### Lectura del gráfico

El gráfico muestra una sola idea:

> **Cómo cambia el desempeño promedio del árbol cuando aumenta su profundidad.**

La profundidad con el mayor R² promedio es la mejor entre las alternativas evaluadas.

Sin embargo, probar manualmente cada valor se vuelve poco práctico cuando existen varios hiperparámetros y muchas combinaciones. Para automatizar este proceso utilizamos **Grid Search**.

# 8. ¿Qué es Grid Search?

Grid Search significa **búsqueda en cuadrícula**.

Primero definimos una lista de valores posibles para cada hiperparámetro.

Ejemplo:

```text
max_depth = [2, 3, 4, 5, 6]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]
```

Grid Search prueba todas las combinaciones posibles.

En este ejemplo existirían:

```text
5 profundidades × 3 valores de min_samples_split × 3 valores de min_samples_leaf
= 45 combinaciones
```

Si utilizamos K-Fold con K = 5, cada combinación se entrena cinco veces:

```text
45 combinaciones × 5 folds = 225 entrenamientos
```

GridSearchCV automatiza todo este procedimiento.

## ¿Qué hace GridSearchCV internamente?

Para cada combinación:

1. Construye el modelo con esos hiperparámetros.
2. Ejecuta K-Fold Cross Validation.
3. Calcula el R² de cada Fold.
4. Calcula el R² promedio.
5. Guarda el resultado.
6. Repite el proceso con la siguiente combinación.
7. Selecciona la combinación con el mayor promedio.

### Idea clave

> GridSearchCV no selecciona el mejor Fold. Selecciona la combinación de hiperparámetros con el mejor desempeño promedio.

# 9. Definición de la cuadrícula

Utilizaremos tres hiperparámetros del árbol:

### `max_depth`

Limita la profundidad máxima.

### `min_samples_split`

Cantidad mínima de observaciones necesarias para dividir un nodo.

Un valor mayor evita divisiones basadas en muy pocos datos.

### `min_samples_leaf`

Cantidad mínima de observaciones que debe contener una hoja final.

Un valor mayor produce árboles menos sensibles a casos individuales.

In [12]:
param_grid = {
    "max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

numero_combinaciones = (
    len(param_grid["max_depth"])
    * len(param_grid["min_samples_split"])
    * len(param_grid["min_samples_leaf"])
)

print(f"Combinaciones que evaluará Grid Search: {numero_combinaciones}")
print(f"Entrenamientos con K=5: {numero_combinaciones * 5}")

Combinaciones que evaluará Grid Search: 72
Entrenamientos con K=5: 360


# 10. Ejecución de GridSearchCV

Usaremos:

- Un árbol de decisión.
- La cuadrícula definida anteriormente.
- K-Fold con K = 5.
- R² como métrica.
- `refit=True`.

`refit=True` indica que, después de encontrar la mejor configuración, GridSearchCV vuelve a entrenar automáticamente un modelo usando:

- Los mejores hiperparámetros.
- Todo el conjunto de entrenamiento.

Ese modelo queda disponible en `best_estimator_`.

In [13]:
arbol = DecisionTreeRegressor(
    random_state=RANDOM_STATE
)

grid_search = GridSearchCV(
    estimator=arbol,
    param_grid=param_grid,
    scoring="r2",
    cv=kfold,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print("Búsqueda finalizada.")

Búsqueda finalizada.


# 11. Mejores resultados

## `best_params_`

Entrega la combinación de hiperparámetros seleccionada.

## `best_score_`

Entrega el mejor R² promedio obtenido mediante validación cruzada.

No corresponde al R² del conjunto de prueba.

In [14]:
print("Mejores hiperparámetros:")
print(grid_search.best_params_)

print(
    f"Mejor R² promedio de validación cruzada: "
    f"{grid_search.best_score_:.3f}"
)

Mejores hiperparámetros:
{'max_depth': 6, 'min_samples_leaf': 1, 'min_samples_split': 2}
Mejor R² promedio de validación cruzada: 0.944


# 12. Revisión de todas las combinaciones

GridSearchCV conserva los resultados completos en `cv_results_`.

Convertiremos esos resultados en una tabla y mostraremos las diez mejores combinaciones.

In [15]:
resultados_grid = pd.DataFrame(
    grid_search.cv_results_
)

columnas_resultados = [
    "rank_test_score",
    "param_max_depth",
    "param_min_samples_split",
    "param_min_samples_leaf",
    "mean_train_score",
    "mean_test_score",
    "std_test_score"
]

top_resultados = (
    resultados_grid[columnas_resultados]
    .sort_values("rank_test_score")
    .head(10)
    .rename(columns={
        "rank_test_score": "Ranking",
        "param_max_depth": "max_depth",
        "param_min_samples_split": "min_samples_split",
        "param_min_samples_leaf": "min_samples_leaf",
        "mean_train_score": "R² promedio entrenamiento",
        "mean_test_score": "R² promedio validación",
        "std_test_score": "Desviación validación"
    })
)

display(
    top_resultados.style.format({
        "R² promedio entrenamiento": "{:.3f}",
        "R² promedio validación": "{:.3f}",
        "Desviación validación": "{:.3f}"
    })
)

,Ranking,max_depth,min_samples_split,min_samples_leaf,R² promedio entrenamiento,R² promedio validación,Desviación validación
36,1,6,2,1,0.999,0.944,0.076
28,2,5,5,1,0.996,0.944,0.076
55,3,10,5,1,0.997,0.944,0.076
64,3,None,5,1,0.997,0.944,0.076
46,5,8,5,1,0.997,0.944,0.076
37,6,6,5,1,0.997,0.944,0.076
27,7,5,2,1,0.998,0.943,0.079
54,8,10,2,1,1.000,0.943,0.073
45,9,8,2,1,1.000,0.942,0.075
63,10,None,2,1,1.000,0.942,0.074


## Interpretación de la tabla

- **Ranking = 1:** mejor combinación.
- **R² promedio entrenamiento:** desempeño promedio sobre los folds de entrenamiento.
- **R² promedio validación:** desempeño promedio sobre los folds reservados.
- **Desviación:** estabilidad de la combinación.

Una gran diferencia entre entrenamiento y validación puede ser una señal de overfitting.

La mejor configuración busca un buen desempeño de validación, no simplemente memorizar el entrenamiento.

# 13. Evaluación del mejor modelo sobre el conjunto de prueba

Hasta ahora, GridSearchCV utilizó únicamente el conjunto de entrenamiento.

Ahora evaluaremos `best_estimator_` sobre el 20% reservado al comienzo.

Esta es la primera vez que el conjunto de prueba participa en el proceso.

In [16]:
mejor_modelo = grid_search.best_estimator_

pred_train_grid = mejor_modelo.predict(X_train)
pred_test_grid = mejor_modelo.predict(X_test)

r2_train_grid = r2_score(y_train, pred_train_grid)
r2_test_grid = r2_score(y_test, pred_test_grid)
mae_test_grid = mean_absolute_error(y_test, pred_test_grid)

print("MEJOR MODELO")
print(f"R² entrenamiento: {r2_train_grid:.3f}")
print(f"R² prueba: {r2_test_grid:.3f}")
print(f"MAE prueba: {mae_test_grid:,.0f}")

MEJOR MODELO
R² entrenamiento: 0.999
R² prueba: 0.593
MAE prueba: 3,982


# 14. Comparación: modelo base vs. modelo optimizado

La optimización no garantiza siempre una mejora enorme en el conjunto de prueba.

Su objetivo es encontrar una configuración evaluada de manera sistemática y reducir decisiones arbitrarias.

In [17]:
comparacion = pd.DataFrame({
    "Modelo": [
        "Árbol base",
        "Árbol optimizado con GridSearchCV"
    ],
    "R² entrenamiento": [
        r2_train_base,
        r2_train_grid
    ],
    "R² prueba": [
        r2_test_base,
        r2_test_grid
    ],
    "MAE prueba": [
        mae_test_base,
        mae_test_grid
    ]
})

display(
    comparacion.style.format({
        "R² entrenamiento": "{:.3f}",
        "R² prueba": "{:.3f}",
        "MAE prueba": "{:,.0f}"
    })
)

,Modelo,R² entrenamiento,R² prueba,MAE prueba
0,Árbol base,1.000,0.603,"3,698"
1,Árbol optimizado con GridSearchCV,0.999,0.593,"3,982"


## Lectura de la comparación

El modelo optimizado puede presentar:

- Menor diferencia entre entrenamiento y prueba.
- Mejor estabilidad.
- Menor complejidad.
- Mejor capacidad de generalización.

No debemos elegir automáticamente el modelo con mayor R² de entrenamiento.

El criterio principal es su desempeño sobre datos no vistos y la consistencia obtenida durante la validación cruzada.

# 15. Entrenamiento del modelo final

Después de:

1. Seleccionar el algoritmo.
2. Encontrar los mejores hiperparámetros.
3. Evaluar el modelo en el conjunto de prueba.

Podemos construir el modelo final utilizando el **100% de los datos disponibles**.

Este modelo final utiliza la configuración seleccionada por GridSearchCV.

In [18]:
mejores_parametros = grid_search.best_params_

modelo_final = DecisionTreeRegressor(
    **mejores_parametros,
    random_state=RANDOM_STATE
)

modelo_final.fit(X, y)

print("Modelo final entrenado con el 100% de los datos.")
print(f"Observaciones utilizadas: {len(X)}")
print("Hiperparámetros utilizados:")
print(mejores_parametros)

Modelo final entrenado con el 100% de los datos.
Observaciones utilizadas: 118
Hiperparámetros utilizados:
{'max_depth': 6, 'min_samples_leaf': 1, 'min_samples_split': 2}


## Importante

El modelo final ya no dispone de un conjunto de prueba interno, porque utilizó todas las observaciones para aprender.

Por eso:

- No debemos presentar su R² de entrenamiento como evidencia de generalización.
- La evidencia de desempeño proviene de la validación cruzada y de la evaluación realizada sobre el conjunto de prueba reservado.

# 16. Flujo completo

```text
Dataset
   │
   ▼
Separación Train/Test
   │
   ▼
GridSearchCV sobre Train
   │
   ├── Prueba combinaciones
   ├── Aplica K-Fold
   ├── Calcula promedios
   └── Selecciona hiperparámetros
   │
   ▼
Evaluación final sobre Test
   │
   ▼
¿Desempeño aceptable?
   │
   ▼
Entrenamiento con el 100% de los datos
   │
   ▼
Modelo final
   │
   ▼
Producción y MLOps
```

# 17. Conclusiones

- Los parámetros se aprenden; los hiperparámetros se configuran.
- Los hiperparámetros controlan la complejidad y el comportamiento del modelo.
- Grid Search prueba todas las combinaciones definidas.
- GridSearchCV utiliza validación cruzada para evaluar cada combinación.
- La selección se basa en el desempeño promedio, no en el mejor Fold.
- El conjunto de prueba se utiliza una sola vez para la evaluación final.
- `best_estimator_` está entrenado con todo el conjunto de entrenamiento.
- Después de validar el modelo, se puede entrenar una versión final con el 100% de los datos.
- El siguiente paso corresponde al despliegue, monitoreo y mantenimiento mediante MLOps.

# Preguntas para discusión

1. ¿Cuál es la diferencia entre un parámetro y un hiperparámetro?
2. ¿Por qué no se debe elegir la profundidad usando directamente el conjunto de prueba?
3. ¿Qué hace K-Fold dentro de GridSearchCV?
4. ¿Qué representa `best_score_`?
5. ¿Por qué `best_score_` no es el R² final de prueba?
6. ¿Qué significa `refit=True`?
7. ¿Por qué no se elige el modelo del mejor Fold?
8. ¿Qué podría ocurrir si la cuadrícula contiene demasiadas combinaciones?
9. ¿Qué ventaja tiene Grid Search frente a probar valores manualmente?
10. ¿Qué modelo debería llevarse finalmente a producción?